# Project FORESIGHT
## Inventory Risk Scoring & Recommended Actions

### Objective

Combine demand forecasts with inventory information to identify SKU-level
inventory risks and recommend appropriate actions.

The notebook will classify SKUs into:

- Stockout Risk
- Overstock Risk
- Healthy

It will also generate recommended inventory actions.

In [104]:
import pandas as pd
import numpy as np

from pathlib import Path

In [105]:
ROOT_DIR = Path.cwd().parent

PROCESSED_DIR = ROOT_DIR / "data" / "processed"

forecast = pd.read_csv(
    PROCESSED_DIR / "forecast_predictions.csv"
)

forecast["week_start"] = pd.to_datetime(
    forecast["week_start"]
)

print("Forecast shape:", forecast.shape)

forecast.head()

Forecast shape: (2400, 4)


,week_start,sku_id,units_sold,prediction
0,2025-10-13,SKU_0001,16,21.475205
1,2025-10-20,SKU_0001,22,27.584302
2,2025-10-27,SKU_0001,26,28.126664
3,2025-11-03,SKU_0001,31,24.259666
4,2025-11-10,SKU_0001,26,22.289730


In [106]:
inventory = pd.read_csv(
    ROOT_DIR / "data" / "raw" / "inventory_snapshots.csv"
)

inventory["date"] = pd.to_datetime(
    inventory["date"]
)

print("Inventory shape:", inventory.shape)

inventory.head()

Inventory shape: (31400, 6)


,date,sku_id,on_hand_units,on_order_units,lead_time_days,reorder_point
0,2023-01-01,SKU_0001,64,0,5,17
1,2023-01-08,SKU_0001,45,0,5,17
2,2023-01-15,SKU_0001,30,0,5,17
3,2023-01-22,SKU_0001,0,36,5,17
4,2023-01-29,SKU_0001,36,0,5,17


In [107]:
latest_inventory = (
    inventory
    .sort_values("date")
    .groupby("sku_id")
    .tail(1)
    .copy()
)

latest_inventory = latest_inventory[
    [
        "sku_id",
        "date",
        "on_hand_units",
        "on_order_units",
        "lead_time_days",
        "reorder_point"
    ]
]

latest_inventory.head()

,sku_id,date,on_hand_units,on_order_units,lead_time_days,reorder_point
31242,SKU_0199,2025-12-28,214,0,8,160
15699,SKU_0100,2025-12-28,77,272,4,108
30928,SKU_0197,2025-12-28,265,0,3,76
10989,SKU_0070,2025-12-28,39,76,10,61
10832,SKU_0069,2025-12-28,38,252,13,186


In [108]:
forecast_summary = (
    forecast
    .groupby("sku_id")
    .agg(
        forecast_12w=("prediction", "sum"),
        avg_weekly_forecast=("prediction", "mean")
    )
    .reset_index()
)

forecast_summary.head()

,sku_id,forecast_12w,avg_weekly_forecast
0,SKU_0001,313.474140,26.122845
1,SKU_0002,450.875042,37.572920
2,SKU_0003,301.584112,25.132009
3,SKU_0004,931.043867,77.586989
4,SKU_0005,1638.156567,136.513047


In [109]:
risk_df = forecast_summary.merge(
    latest_inventory,
    on="sku_id",
    how="left"
)

risk_df.head()

,sku_id,forecast_12w,avg_weekly_forecast,date,on_hand_units,on_order_units,lead_time_days,reorder_point
0,SKU_0001,313.474140,26.122845,2025-12-28,16,44,5,17
1,SKU_0002,450.875042,37.572920,2025-12-28,66,0,14,54
2,SKU_0003,301.584112,25.132009,2025-12-28,36,0,6,19
3,SKU_0004,931.043867,77.586989,2025-12-28,0,121,9,97
4,SKU_0005,1638.156567,136.513047,2025-12-28,94,241,9,156


In [110]:
sku_master = pd.read_csv(
    ROOT_DIR / "data" / "raw" / "sku_master.csv"
)

risk_df = risk_df.merge(
    sku_master[
        ["sku_id", "category", "subcategory", "unit_cost", "list_price"]
    ],
    on="sku_id",
    how="left"
)

risk_df.head()

,sku_id,forecast_12w,avg_weekly_forecast,date,on_hand_units,on_order_units,lead_time_days,reorder_point,category,subcategory,unit_cost,list_price
0,SKU_0001,313.474140,26.122845,2025-12-28,16,44,5,17,Furniture,Desk,6911.20,14977.29
1,SKU_0002,450.875042,37.572920,2025-12-28,66,0,14,54,Decor,Lamp,7812.29,17478.07
2,SKU_0003,301.584112,25.132009,2025-12-28,36,0,6,19,Kitchen,Utensil,3767.97,6812.03
3,SKU_0004,931.043867,77.586989,2025-12-28,0,121,9,97,Furniture,Chair,5257.76,12119.34
4,SKU_0005,1638.156567,136.513047,2025-12-28,94,241,9,156,Kitchen,Appliance,4570.30,6719.25


In [111]:
lead_time_demand = (
    risk_df["avg_weekly_forecast"]
    * risk_df["lead_time_days"]
    / 7
)

risk_df["lead_time_demand"] = lead_time_demand

risk_df["stockout_units_at_risk"] = (
    lead_time_demand
    - risk_df["on_hand_units"]
).clip(lower=0)

risk_df["stockout_sales_at_risk"] = (
    risk_df["stockout_units_at_risk"]
    * risk_df["list_price"]
)

# Target 12-week inventory level
target_inventory_units = (
    risk_df["avg_weekly_forecast"] * 12
)

risk_df["excess_units"] = (
    risk_df["on_hand_units"]
    - target_inventory_units
).clip(lower=0)

risk_df["overstock_capital_locked"] = (
    risk_df["excess_units"]
    * risk_df["unit_cost"]
)

In [112]:
risk_df["weeks_of_cover"] = np.where(
    risk_df["avg_weekly_forecast"] > 0,
    (
        risk_df["on_hand_units"]
        + risk_df["on_order_units"]
    )
    / risk_df["avg_weekly_forecast"],
    np.inf
)

risk_df["weeks_of_cover"] = risk_df[
    "weeks_of_cover"
].round(2)

risk_df.head()

,sku_id,forecast_12w,avg_weekly_forecast,date,on_hand_units,on_order_units,lead_time_days,reorder_point,category,subcategory,unit_cost,list_price,lead_time_demand,stockout_units_at_risk,stockout_sales_at_risk,excess_units,overstock_capital_locked,weeks_of_cover
0,SKU_0001,313.474140,26.122845,2025-12-28,16,44,5,17,Furniture,Desk,6911.20,14977.29,18.659175,2.659175,3.982724e+04,0.0,0.0,2.30
1,SKU_0002,450.875042,37.572920,2025-12-28,66,0,14,54,Decor,Lamp,7812.29,17478.07,75.145840,9.145840,1.598516e+05,0.0,0.0,1.76
2,SKU_0003,301.584112,25.132009,2025-12-28,36,0,6,19,Kitchen,Utensil,3767.97,6812.03,21.541722,0.000000,0.000000e+00,0.0,0.0,1.43
3,SKU_0004,931.043867,77.586989,2025-12-28,0,121,9,97,Furniture,Chair,5257.76,12119.34,99.754700,99.754700,1.208961e+06,0.0,0.0,1.56
4,SKU_0005,1638.156567,136.513047,2025-12-28,94,241,9,156,Kitchen,Appliance,4570.30,6719.25,175.516775,81.516775,5.477316e+05,0.0,0.0,2.45


In [113]:
risk_df["risk_category"] = np.select(
    [
        risk_df["stockout_units_at_risk"] > 0,

        (
            risk_df["weeks_of_cover"] > 12
        )
        &
        (
            risk_df["excess_units"] > 0
        )
    ],
    [
        "Stockout Risk",
        "Overstock Risk"
    ],
    default="Healthy"
)

risk_df["risk_category"].value_counts()

risk_category
Healthy          105
Stockout Risk     95
Name: count, dtype: int64

In [114]:
risk_df["recommended_action"] = np.select(
    [
        risk_df["risk_category"] == "Stockout Risk",
        risk_df["risk_category"] == "Overstock Risk"
    ],
    [
        "Prioritize replenishment",
        "Reduce or defer replenishment"
    ],
    default="Maintain current inventory"
)

In [115]:
impact_summary = pd.DataFrame({
    "metric": [
        "Total potential stockout sales at risk",
        "Total capital locked in overstock"
    ],
    "value_inr": [
        risk_df["stockout_sales_at_risk"].sum(),
        risk_df["overstock_capital_locked"].sum()
    ]
})

impact_summary

,metric,value_inr
0,Total potential stockout sales at risk,7.491323e+07
1,Total capital locked in overstock,0.000000e+00


In [116]:
risk_summary = (
    risk_df["risk_category"]
    .value_counts()
    .rename_axis("risk_category")
    .reset_index(name="sku_count")
)

risk_summary

,risk_category,sku_count
0,Healthy,105
1,Stockout Risk,95


In [117]:
priority_skus = (
    risk_df
    .sort_values(
        [
            "risk_category",
            "weeks_of_cover"
        ],
        ascending=[True, True]
    )
)

priority_skus[
    [
        "sku_id",
        "forecast_12w",
        "on_hand_units",
        "on_order_units",
        "lead_time_days",
        "reorder_point",
        "weeks_of_cover",
        "risk_category",
        "recommended_action"
    ]
].head(20)

,sku_id,forecast_12w,on_hand_units,on_order_units,lead_time_days,reorder_point,weeks_of_cover,risk_category,recommended_action
137,SKU_0138,2911.160338,105,0,3,85,0.43,Healthy,Maintain current inventory
189,SKU_0190,705.517622,34,0,3,20,0.58,Healthy,Maintain current inventory
136,SKU_0137,2794.306188,172,0,5,141,0.74,Healthy,Maintain current inventory
131,SKU_0132,1760.253988,112,0,4,81,0.76,Healthy,Maintain current inventory
163,SKU_0164,2230.139858,146,0,5,123,0.79,Healthy,Maintain current inventory
30,SKU_0031,1785.499253,122,0,4,77,0.82,Healthy,Maintain current inventory
139,SKU_0140,461.784820,35,0,4,19,0.91,Healthy,Maintain current inventory
33,SKU_0034,4275.122740,340,0,5,242,0.95,Healthy,Maintain current inventory
57,SKU_0058,555.588841,45,0,4,22,0.97,Healthy,Maintain current inventory
67,SKU_0068,2516.161399,212,0,6,148,1.01,Healthy,Maintain current inventory


In [118]:
risk_df[
    [
        "sku_id",
        "on_hand_units",
        "on_order_units",
        "avg_weekly_forecast",
        "forecast_12w",
        "weeks_of_cover",
        "lead_time_days",
        "lead_time_demand",
        "stockout_units_at_risk",
        "stockout_sales_at_risk"
    ]
].sort_values(
    "stockout_sales_at_risk",
    ascending=False
).head(10)

,sku_id,on_hand_units,on_order_units,avg_weekly_forecast,forecast_12w,weeks_of_cover,lead_time_days,lead_time_demand,stockout_units_at_risk,stockout_sales_at_risk
47,SKU_0048,128,1074,560.673411,6728.080932,2.14,7,560.673411,432.673411,6.625312e+06
102,SKU_0103,258,769,377.147175,4525.766104,2.72,13,700.416183,442.416183,5.273716e+06
114,SKU_0115,51,322,149.325367,1791.904401,2.50,13,277.318538,226.318538,3.769668e+06
28,SKU_0029,421,697,351.140070,4213.680838,3.18,13,652.117273,231.117273,3.658614e+06
93,SKU_0094,0,466,341.380194,4096.562329,1.37,8,390.148793,390.148793,3.600071e+06
133,SKU_0134,152,292,225.120261,2701.443127,1.97,13,418.080484,266.080484,3.137406e+06
31,SKU_0032,154,447,313.824905,3765.898856,1.92,14,627.649809,473.649809,2.927753e+06
68,SKU_0069,38,252,123.248272,1478.979260,2.35,13,228.889647,190.889647,2.905730e+06
143,SKU_0144,40,664,265.873665,3190.483982,2.65,6,227.891713,187.891713,2.713947e+06
24,SKU_0025,139,294,185.130824,2221.569884,2.34,13,343.814387,204.814387,2.290931e+06


In [119]:
output_path = (
    PROCESSED_DIR
    / "inventory_risk_scores.csv"
)

risk_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(risk_df))

Saved: c:\Users\saqib\Desktop\zidio\foresight-demand-inventory\data\processed\inventory_risk_scores.csv
Rows: 200


# Conclusion

The inventory risk layer combines demand forecasts with current inventory,
on-order inventory, lead time, and reorder points to identify SKU-level
inventory risks.

Each SKU receives:

- Forecasted 12-week demand
- Average weekly forecast
- Lead-time demand
- Weeks of inventory cover
- Stockout units at risk
- Potential stockout sales at risk (₹)
- Excess inventory units
- Overstock capital locked (₹)
- Risk category
- Recommended action

The final risk assessment classified 200 SKUs as:

- 105 Healthy
- 95 Stockout Risk
- 0 Overstock Risk

The estimated potential stockout sales at risk are approximately
₹7.49 crore. No overstock capital was identified in the current dataset
under the defined 12-week inventory threshold.

The resulting `inventory_risk_scores.csv` is used by the decision-support
dashboard and recommendation layer.